# PricePoint — 09 · Tune Gradient Boosting

**Why tune:** a model has settings (hyperparameters) that we choose, not the computer. So far we used one hand-picked set. Here we **try many combinations** and keep the best one.

**How:** `GridSearchCV` tries every combination in a small grid. For each one it trains 5 times on different slices of the **training** data and averages the error (MAE). The combination with the lowest MAE wins. The test set is not touched while searching.

**What we tune:**
- `n_estimators` — how many small trees to build
- `learning_rate` — how big a step each new tree takes (small = careful)
- `max_depth` — how complex each tree can be
- `subsample` — the share of laptops each tree sees (less than 1.0 adds a little randomness, which helps against overfitting)

In [1]:
import json

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

In [2]:

ML_FOLDER = None

candidates = [Path(".."), Path("."), Path("ml")]
if ML_FOLDER is not None:
    candidates.insert(0, Path(ML_FOLDER))

for base in candidates:
    if (base / "data" / "laptop_price_clean.csv").exists():
        DATA_DIR = base / "data"
        MODEL_DIR = base / "models"
        break
else:
    raise FileNotFoundError(
        "Cannot find data/laptop_price_clean.csv. Move this notebook into PricePoint/ml/notebooks, "
        "or set ML_FOLDER in this cell to the path of your project's ml folder."
    )

MODEL_DIR.mkdir(exist_ok=True)

In [3]:
# --- load the clean data ---

clean = pd.read_csv(DATA_DIR / "laptop_price_clean.csv")


clean["touchscreen"] = clean["touchscreen"].astype(int)
clean["ips"] = clean["ips"].astype(int)

# the columns the models will use
categorical_columns = ["brand", "type", "cpu_brand", "storage_type", "gpu_brand", "os"]
numeric_columns = [
    "inches",
    "touchscreen",
    "ips",
    "resolution_pixels",
    "cpu_speed_ghz",
    "ram_gb",
    "storage_gb",
    "weight_kg",
]

X = clean[categorical_columns + numeric_columns]
y = clean["price_bdt"]

print("rows:", len(X))
print("features:", X.shape[1])

rows: 1303
features: 14


## Split into training and testing

In [4]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("training rows:", len(X_train))
print("testing rows :", len(X_test))

training rows: 1042
testing rows : 261


## Build the pipeline

In [5]:

preprocessor = ColumnTransformer(
    transformers=[
        (
            "category",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
        ("number", "passthrough", numeric_columns),
    ]
)

In [6]:

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42)),
])

## The search grid

3 × 2 × 3 × 2 = **36 combinations**, each trained 5 times = 180 trainings. This takes a minute or two.

In [7]:

param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [3, 4, 5],
    "model__subsample": [0.85, 1.0],
}

search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
search.fit(X_train, y_train)

print("best settings:", search.best_params_)
print("best CV MAE  :", round(-search.best_score_), "BDT")

best settings: {'model__learning_rate': 0.1, 'model__max_depth': 5, 'model__n_estimators': 300, 'model__subsample': 1.0}
best CV MAE  : 22110 BDT


### The 5 best combinations

In [8]:

top = pd.DataFrame(search.cv_results_).sort_values("rank_test_score").head(5)
top = top[["param_model__n_estimators", "param_model__learning_rate",
           "param_model__max_depth", "param_model__subsample", "mean_test_score"]].copy()
top["cv_mae"] = (-top["mean_test_score"]).round()
top = top.drop(columns="mean_test_score")
top.columns = ["trees", "learning_rate", "max_depth", "subsample", "cv_mae"]
print(top.to_string(index=False))

 trees  learning_rate  max_depth  subsample  cv_mae
   300           0.10          5       1.00 22110.0
   200           0.10          5       1.00 22132.0
   300           0.05          5       0.85 22171.0
   300           0.10          4       0.85 22271.0
   200           0.05          5       0.85 22306.0


## Is the tuned model really better?

We compare it with the settings from notebook 07 on the **same hidden test laptops**.

In [9]:

default_settings = {
    "model__n_estimators": 120,
    "model__learning_rate": 0.08,
    "model__max_depth": 4,
    "model__subsample": 0.85,
}

default_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42)),
]).set_params(**default_settings)
default_pipeline.fit(X_train, y_train)

pipeline = search.best_estimator_         

for name, fitted in (("default (notebook 07)", default_pipeline), ("tuned", pipeline)):
    y_pred = fitted.predict(X_test)
    print(f"{name:<22} test MAE = {mean_absolute_error(y_test, y_pred):>7,.0f} BDT   "
          f"test R² = {r2_score(y_test, y_pred):.3f}")

default (notebook 07)  test MAE =  21,398 BDT   test R² = 0.822
tuned                  test MAE =  21,995 BDT   test R² = 0.810


**How to read this:** the search picks the combination with the best *cross-validation* error on the training data, but what matters is the hidden test set. If the tuned model is about the same, or even slightly worse, that is a normal and honest result: the hand-picked settings were already close to the best, and with only 261 test laptops small differences are partly luck. Tuning cannot create information the data does not contain.

A tuned model that fits the training data even more closely (higher training R²) can also be a sign it is starting to overfit.

## Score it

Three numbers, same meaning everywhere in this project:

- **MAE** — average miss in taka ("on average we are off by X BDT")
- **RMSE** — like MAE, but big mistakes are punished harder
- **R²** — how much of the price spread the model explains (1.0 = perfect, 0 = useless)

We score twice: on the held-out **test set**, and with **5-fold cross-validation** (more reliable — the data is shuffled and scored 5 different ways).

In [10]:
# --- score on the test set ---

y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
r2_train = pipeline.score(X_train, y_train)

print("R² on training data :", round(r2_train, 3))
print("R² on test data     :", round(r2, 3))
print("MAE                 :", round(mae), "BDT")
print("RMSE                :", round(rmse), "BDT")

R² on training data : 0.989
R² on test data     : 0.81
MAE                 : 21995 BDT
RMSE                : 38798 BDT


## Cross-validation

Shuffle the data, train 5 times on different slices, average the scores.

In [11]:

cv_mae = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_mean_absolute_error")
cv_rmse = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_root_mean_squared_error")
cv_r2 = cross_val_score(pipeline, X, y, cv=5, scoring="r2")

print("CV MAE :", round(cv_mae.mean()), "±", round(cv_mae.std()))
print("CV RMSE:", round(cv_rmse.mean()), "±", round(cv_rmse.std()))
print("CV R²  :", round(cv_r2.mean(), 3), "±", round(cv_r2.std(), 3))

CV MAE : 24595 ± 1166
CV RMSE: 38979 ± 3694
CV R²  : 0.796 ± 0.03


## Try it on 3 real laptops

In [12]:

sample = X_test.head(3)
sample_predicted = pipeline.predict(sample)

for position in range(3):
    real_price = y_test.iloc[position]
    predicted_price = round(sample_predicted[position])
    print("real:", real_price, "BDT    predicted:", predicted_price, "BDT")

real: 209000 BDT    predicted: 145550 BDT
real: 143625 BDT    predicted: 127372 BDT
real: 62375 BDT    predicted: 53840 BDT


## Save the scores and the best settings

- The scores go to `models/metrics_gradient_boosting_tuned.json`, so `08_compare_models.ipynb` shows the tuned model as its own row.
- The settings go to `models/best_gb_params.json`, for reference.

Only if the tuned model is clearly better on the test set **and** in cross-validation should you copy these settings into `src/train_and_export.py` so the website uses them. If it is not clearly better, keep the current settings.

In [13]:


metrics = {
    "model": "Gradient Boosting (tuned)",
    "test_mae": round(mae, 2),
    "test_rmse": round(rmse, 2),
    "test_r2": round(r2, 4),
    "cv_mae": round(cv_mae.mean(), 2),
    "cv_rmse": round(cv_rmse.mean(), 2),
    "cv_r2": round(cv_r2.mean(), 4),
}

metrics_path = MODEL_DIR / "metrics_gradient_boosting_tuned.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print("saved:", metrics_path)


best_settings = {key.replace("model__", ""): value for key, value in search.best_params_.items()}
params_path = MODEL_DIR / "best_gb_params.json"
with open(params_path, "w") as f:
    json.dump(best_settings, f, indent=2)
print("saved:", params_path, best_settings)

saved: ../models/metrics_gradient_boosting_tuned.json
saved: ../models/best_gb_params.json {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300, 'subsample': 1.0}


**Next notebook:** `10_error_analysis.ipynb`